In [1]:
%%writefile tokenizer.py
"""
Shared tokenizer + vocabulary module for NLP Assignment 2, Project 1 (DailyDialog).

Used identically by Task 1 (n-grams), Task 2 (AWD-LSTM), Task 3 (hybrid), and the
BLiMP evaluation harness. Build the vocab ONCE from DailyDialog's train split and
reuse the same Vocab object everywhere — never rebuild it per-task.

Design choices :
- Punctuation-aware whitespace tokenizer: splits words from attached punctuation
  (e.g. "hello!" -> ["hello", "!"]) but does not split contractions internally
  (e.g. "don't" stays as ["don't"]) since the spec doesn't ask for that level of
  linguistic splitting and DailyDialog is casual/contracted speech.
- Lowercase everything.
- Vocabulary built from train only, capped at 30,000 most frequent tokens.
  Tokens occurring <2 times in train, and any OOV token seen later, map to <UNK>.
- Special tokens: <unk>, <s>, </s>, <pad> (pad reserved for Task 2 batching; not
  used by Task 1's counting).
"""

import re
import json
from collections import Counter
from typing import List, Dict, Iterable


# ---------------------------------------------------------------------------
# Tokenization
# ---------------------------------------------------------------------------

# Matches: contractions/words with internal apostrophes (don't, it's) as one
# token, plain word tokens, or any single punctuation/symbol character.
_TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+(?:\.[0-9]+)?|[^\sA-Za-z0-9]")


def _normalize_apostrophes(text: str) -> str:
    """Collapse all Unicode apostrophe/quote variants to a single straight
    ASCII apostrophe (') before any other processing.

    Real DailyDialog text mixes the straight apostrophe (U+0027, ') with the
    Unicode "right single quotation mark" (U+2019, ’) and occasionally other
    look-alikes, inconsistently -- sometimes within the same split. Left
    unnormalized, contraction detection silently fails for every curly-quote
    instance (e.g. "I \u2019 m here" would NOT be caught by the spacing-fix
    regex below, since that regex only matches the ASCII character), leaking
    junk single-character tokens ('m', 't', 's', etc.) into the vocabulary.
    This must run BEFORE `_normalize_spacing`.
    """
    return text.replace("\u2019", "'").replace("\u2018", "'").replace("`", "'")


def _normalize_spacing(text: str) -> str:
    """Fix known DailyDialog spacing artifacts before tokenizing.

    Real DailyDialog utterances frequently have spaces on BOTH sides of the
    apostrophe inside a contraction, e.g. "Let ' s go" instead of "Let's go".
    Left alone, this breaks contraction detection entirely: the word, the
    apostrophe, and the suffix end up as three spurious separate tokens
    ("let", "'", "s") instead of one "let's" token, polluting the vocabulary
    with junk single-character tokens. This regex collapses "<word> ' <sfx>"
    into "<word>'<sfx>" whenever the trailing part looks like a common
    contraction suffix (s, t, re, ve, ll, d, m), so "let ' s" -> "let's",
    "they ' re" -> "they're", etc. Requires a preceding word character so it
    doesn't fire on a bare leading apostrophe/quote mark.

    This is a documented, deliberate preprocessing deviation -- state it in
    the report's preprocessing subsection.
    """
    text = re.sub(
        r"(\w)\s*'\s+(s|t|re|ve|ll|d|m)\b",
        r"\1'\2",
        text,
        flags=re.IGNORECASE,
    )
    return text


def tokenize(text: str) -> List[str]:
    """Punctuation-aware whitespace tokenizer + lowercase.

    Applies DailyDialog-specific spacing normalization (see
    `_normalize_spacing`) before the regex tokenizer runs.

    Example:
        >>> tokenize("Hello! Don't you think it's 3.5 degrees out?")
        ['hello', '!', "don't", 'you', 'think', "it's", '3.5', 'degrees', 'out', '?']
        >>> tokenize("Good . Let ' s go now .")
        ['good', '.', "let's", 'go', 'now', '.']
        >>> tokenize("I \u2019 m here .")
        ["i'm", 'here', '.']
    """
    text = _normalize_apostrophes(text)
    text = _normalize_spacing(text)
    text = text.lower()
    return _TOKEN_RE.findall(text)


def join_dialogue(utterances: List[str]) -> str:
    """Join a DailyDialog `dialog` list into one document.

    Per spec: concatenate utterances in order, separated by a single space.
    NO speaker tags, no turn markers.
    """
    return " ".join(u.strip() for u in utterances)


# ---------------------------------------------------------------------------
# Vocabulary
# ---------------------------------------------------------------------------

UNK = "<unk>"
BOS = "<s>"
EOS = "</s>"
PAD = "<pad>"

SPECIAL_TOKENS = [PAD, UNK, BOS, EOS]  # fixed order -> fixed ids 0,1,2,3


class Vocab:
    """Fixed vocabulary built from a token frequency count.

    Attributes:
        token2id: dict token -> int id
        id2token: list, index = id
        freq: Counter of raw training-set token frequencies (pre-cap, pre-UNK
              collapse) -- kept for diagnostics / report stats, not used at
              runtime.
    """

    def __init__(self, token2id: Dict[str, int], id2token: List[str], freq: Counter):
        self.token2id = token2id
        self.id2token = id2token
        self.freq = freq

    def __len__(self):
        return len(self.id2token)

    def encode(self, tokens: Iterable[str]) -> List[int]:
        unk_id = self.token2id[UNK]
        return [self.token2id.get(t, unk_id) for t in tokens]

    def encode_str(self, tokens: Iterable[str]) -> List[str]:
        """Map OOV tokens to the literal '<unk>' string (used by n-gram code
        that works with token strings rather than ids)."""
        vocab_set = set(self.token2id.keys())
        return [t if t in vocab_set else UNK for t in tokens]

    def decode(self, ids: Iterable[int]) -> List[str]:
        return [self.id2token[i] for i in ids]

    def save(self, path: str):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "id2token": self.id2token,
                    "freq": dict(self.freq),
                },
                f,
                ensure_ascii=False,
                indent=1,
            )

    @classmethod
    def load(cls, path: str) -> "Vocab":
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        id2token = obj["id2token"]
        token2id = {t: i for i, t in enumerate(id2token)}
        freq = Counter(obj["freq"])
        return cls(token2id, id2token, freq)


def build_vocab(
    train_token_lists: Iterable[List[str]],
    max_size: int = 30000,
    min_count: int = 2,
) -> Vocab:
    """Build a Vocab from an iterable of already-tokenized training documents.

    Args:
        train_token_lists: iterable where each element is the token list for
            one training document (one DailyDialog dialogue, already joined +
            tokenized). Pass the full train split here, nothing held out.
        max_size: cap on vocabulary size INCLUDING special tokens. Spec says
            "30,000 most frequent tokens" -- we interpret this as 30,000
            regular (non-special) tokens, with the 4 special tokens added on
            top, since the special tokens are not part of the natural-language
            frequency count the spec is capping. State this in the report if
            asked; trivial to change `max_size` semantics here if TA disagrees.
        min_count: tokens occurring fewer than this many times in train are
            collapsed into <UNK> even if they'd otherwise fit under max_size.

    Returns:
        Vocab object. Deterministic tie-breaking: when multiple tokens have
        identical frequency at the cutoff boundary, ties are broken
        alphabetically (ascending) for full reproducibility across runs/seeds.
    """
    freq = Counter()
    for tokens in train_token_lists:
        freq.update(tokens)

    # Keep tokens meeting min_count, then cap to max_size most frequent,
    # alphabetical tie-break for determinism.
    eligible = [(tok, c) for tok, c in freq.items() if c >= min_count]
    eligible.sort(key=lambda x: (-x[1], x[0]))  # freq desc, then alpha asc
    kept = [tok for tok, _ in eligible[:max_size]]

    id2token = list(SPECIAL_TOKENS) + kept
    token2id = {t: i for i, t in enumerate(id2token)}

    return Vocab(token2id, id2token, freq)


# ---------------------------------------------------------------------------
# Scoring-stream builder (shared by Task 1 n-gram eval AND Task 2 LSTM eval,
# so Task 3's per-position interpolation is well-defined -- see PLAN.md 3.7)
# ---------------------------------------------------------------------------

def build_scoring_stream(
    dialogues: List[List[str]],
    vocab: Vocab,
) -> List[str]:
    """Turn a list of dialogues (each a list of raw utterance strings) into
    ONE flat list of vocab-mapped token strings, with <s> ... </s> wrapped
    around each dialogue (per-dialogue boundaries, NOT per-utterance -- see
    PLAN.md assumption 3.2).

    This exact function must be used by both the n-gram scorer and the LSTM
    scorer at evaluation time so both models score an identical sequence of
    positions. Do not reimplement this logic separately in task2/task3 code.
    """
    stream: List[str] = []
    for utterances in dialogues:
        doc = join_dialogue(utterances)
        toks = tokenize(doc)
        toks = vocab.encode_str(toks)
        stream.append(BOS)
        stream.extend(toks)
        stream.append(EOS)
    return stream


if __name__ == "__main__":
    # Minimal smoke test using fabricated data (NOT real DailyDialog -- just
    # verifies the module's internal logic before we touch Colab/real data).
    fake_dialogues = [
        ["Hello there!", "Hi, how are you?", "I'm fine, thanks."],
        ["Don't forget the meeting.", "I won't. See you at 3.5pm!"],
        ["Hello there!", "Hi, how are you?"],  # repeat to test frequency counts
    ]

    tokenized_docs = [tokenize(join_dialogue(d)) for d in fake_dialogues]
    for d, toks in zip(fake_dialogues, tokenized_docs):
        print(join_dialogue(d), "->", toks)

    vocab = build_vocab(tokenized_docs, max_size=30000, min_count=2)
    print("\nVocab size (incl. specials):", len(vocab))
    print("Tokens kept:", [t for t in vocab.id2token if t not in SPECIAL_TOKENS])
    print("'hello' in vocab:", "hello" in vocab.token2id)
    print("'meeting' in vocab (freq 1, should be UNK'd):", "meeting" in vocab.token2id)

    stream = build_scoring_stream(fake_dialogues, vocab)
    print("\nScoring stream:")
    print(stream)

    # Save/load round trip
    vocab.save("/tmp/test_vocab.json")
    reloaded = Vocab.load("/tmp/test_vocab.json")
    assert reloaded.id2token == vocab.id2token
    print("\nSave/load round-trip OK.")

Writing tokenizer.py


In [2]:
%%writefile ngram_lm.py
"""
N-gram language model: count tables + 5 smoothing variants (Task 1).

Shared counting + 3 smoothing families:
  - Add-k (k in {1, 0.5, 0.1})  -- 3 variants, same formula, different k
  - Good-Turing (r <= 5 re-estimated, backoff for r=0 -- see PLAN.md 3.3)
  - Interpolated Kneser-Ney (d=0.75 fixed, + outer lambda blend w/ unigram,
    lambda grid-searched on validation -- see PLAN.md 3.4)

Design: build ONE set of n-gram count tables per order (1..4) from the train
scoring-stream, then every smoothing method reads from those same tables.
Never recount per smoothing variant.

All models expose the same two methods so Task 1's PPL loop, Task 3's hybrid,
and the BLiMP scorer can all treat them interchangeably:
  - logprob(context: tuple[str,...], word: str) -> float   (natural log)
  - a `order` attribute (the n in n-gram, e.g. 3 for trigram)
"""

import math
from collections import defaultdict, Counter
from typing import List, Tuple, Dict


# ---------------------------------------------------------------------------
# Count tables
# ---------------------------------------------------------------------------

class NgramCounts:
    """Count tables for orders 1..max_order, built from one token stream.

    counts[n] is a dict: n-gram tuple (length n) -> count
    context_counts[n] is a dict: (n-1)-gram tuple -> total count of that
        context (i.e. sum of counts[n] over all n-grams sharing that prefix)
        -- stored separately to avoid recomputing sums repeatedly.
    """

    def __init__(self, stream: List[str], max_order: int = 4):
        self.max_order = max_order
        self.counts: Dict[int, Dict[Tuple[str, ...], int]] = {
            n: defaultdict(int) for n in range(1, max_order + 1)
        }
        self.vocab_tokens = set()

        for n in range(1, max_order + 1):
            for i in range(len(stream) - n + 1):
                ngram = tuple(stream[i:i + n])
                self.counts[n][ngram] += 1

        self.vocab_tokens = set(stream)
        self.vocab_size = len(self.vocab_tokens)

        # context_counts[n][context] = sum over all words w of counts[n][context + (w,)]
        # equivalently: context_counts[n] == counts[n-1] restricted to contexts
        # that actually appear as an (n-1)-gram -- but safer to just recompute
        # from counts[n] directly so it's always consistent with counts[n].
        self.context_counts: Dict[int, Dict[Tuple[str, ...], int]] = {
            n: defaultdict(int) for n in range(1, max_order + 1)
        }
        for n in range(1, max_order + 1):
            for ngram, c in self.counts[n].items():
                context = ngram[:-1]
                self.context_counts[n][context] += c

        # Count-of-counts tables per order, for Good-Turing: N_r = number of
        # distinct n-grams of this order that occurred exactly r times.
        self.count_of_counts: Dict[int, Counter] = {}
        for n in range(1, max_order + 1):
            self.count_of_counts[n] = Counter(self.counts[n].values())


# ---------------------------------------------------------------------------
# Add-k smoothing
# ---------------------------------------------------------------------------

class AddKModel:
    """Add-k (Laplace generalization) smoothing for a fixed n-gram order."""

    def __init__(self, ngram_counts: NgramCounts, order: int, k: float):
        self.nc = ngram_counts
        self.order = order
        self.k = k
        self.V = ngram_counts.vocab_size

    def logprob(self, context: Tuple[str, ...], word: str) -> float:
        context = context[-(self.order - 1):] if self.order > 1 else ()
        ngram = context + (word,)
        c_ngram = self.nc.counts[self.order].get(ngram, 0)
        c_context = self.nc.context_counts[self.order].get(context, 0)
        prob = (c_ngram + self.k) / (c_context + self.k * self.V)
        return math.log(prob)


# ---------------------------------------------------------------------------
# Good-Turing smoothing
# ---------------------------------------------------------------------------

class GoodTuringModel:
    """Good-Turing discounting for r <= gt_max; backoff to (order-1) MLE for
    unseen (r=0) n-grams (see PLAN.md assumption 3.3).

    Adjusted count: r* = (r+1) * N_{r+1} / N_r  for r in [1, gt_max]
    For r=0 (unseen n-gram): no direct GT estimate exists (N_0 is not a count
    we observe in the usual formula), so we fall back to a simple backoff:
    distribute a held-out mass (estimated as N_1 / N_total, i.e. the
    Good-Turing estimate for total unseen mass) proportionally via the
    (order-1)-gram MLE distribution. This is a standard simple Katz-style
    backoff, not full Katz backoff with its own discounting at every level --
    kept deliberately simple per the project's "minimal effort, ship it"
    principle, documented explicitly in the report.
    """

    def __init__(self, ngram_counts: NgramCounts, order: int, gt_max: int = 5):
        self.nc = ngram_counts
        self.order = order
        self.gt_max = gt_max
        self.V = ngram_counts.vocab_size

        coc = ngram_counts.count_of_counts[order]
        self.N_total = sum(r * n for r, n in coc.items())  # total token count at this order

        # Precompute adjusted counts r* for r = 1..gt_max
        self.r_star = {}
        for r in range(1, gt_max + 1):
            Nr = coc.get(r, 0)
            Nr1 = coc.get(r + 1, 0)
            if Nr == 0:
                self.r_star[r] = float(r)  # no data to adjust, fall back to raw count
            else:
                self.r_star[r] = (r + 1) * Nr1 / Nr if Nr1 > 0 else float(r)

        # Mass reserved for unseen n-grams (r=0): N_1 / N_total (standard GT)
        N1 = coc.get(1, 0)
        self.unseen_mass = (N1 / self.N_total) if self.N_total > 0 else 1e-10

        # Lower-order model for backing off unseen n-grams (order-1 MLE,
        # or uniform if order==1)
        if order > 1:
            self._lower = GoodTuringModel(ngram_counts, order - 1, gt_max) \
                if order - 1 >= 1 else None
        else:
            self._lower = None

    def _raw_count(self, context, word):
        ngram = context + (word,)
        return self.nc.counts[self.order].get(ngram, 0)

    def logprob(self, context: Tuple[str, ...], word: str) -> float:
        context = context[-(self.order - 1):] if self.order > 1 else ()
        r = self._raw_count(context, word)
        c_context = self.nc.context_counts[self.order].get(context, 0)

        if r == 0:
            # Backoff: unseen_mass spread over OOV-given-context, approximated
            # via lower-order distribution (or uniform at unigram level).
            if self.order == 1 or self._lower is None:
                prob = self.unseen_mass / max(self.V, 1)
            else:
                lower_context = context[1:] if len(context) > 0 else ()
                lower_logprob = self._lower.logprob(lower_context, word)
                prob = self.unseen_mass * math.exp(lower_logprob)
            prob = max(prob, 1e-12)  # floor to avoid log(0)
            return math.log(prob)
        elif r <= self.gt_max:
            r_adj = self.r_star[r]
            prob = r_adj / c_context if c_context > 0 else r_adj / max(self.V, 1)
            return math.log(max(prob, 1e-12))
        else:
            # r > gt_max: use raw MLE, undiscounted (standard GT practice --
            # only low counts are unreliable enough to need discounting)
            prob = r / c_context
            return math.log(prob)


# ---------------------------------------------------------------------------
# Interpolated Kneser-Ney (+ outer unigram blend, per PLAN.md 3.4)
# ---------------------------------------------------------------------------

class KneserNeyModel:
    """Standard recursive interpolated KN with fixed discount d, PLUS an
    outer global interpolation with the unigram distribution:

        P_final(w | context) = lam * P_KN(w | context) + (1-lam) * P_unigram(w)

    lam is tuned by the caller via grid search on validation perplexity (see
    `tune_kn_lambda` below) and passed in at construction time.

    See PLAN.md assumption 3.4 for why this two-part structure was chosen to
    resolve the spec's ambiguous "d=0.75 AND separately tune lambda" wording.
    """

    def __init__(self, ngram_counts: NgramCounts, order: int, d: float = 0.75,
                 lam: float = 1.0):
        self.nc = ngram_counts
        self.order = order
        self.d = d
        self.lam = lam
        self.V = ngram_counts.vocab_size

        # Precompute unigram MLE distribution for the outer blend.
        unigram_counts = ngram_counts.counts[1]
        total_unigrams = sum(unigram_counts.values())
        self._unigram_prob = {
            w: c / total_unigrams for (w,), c in unigram_counts.items()
        }
        self._unigram_floor = 1e-10  # for words with zero unigram count (shouldn't happen if w in vocab)

        # Precompute continuation counts for true KN lower-order smoothing:
        # continuation count of a word w at order n = number of distinct
        # (n)-gram contexts of order (n+1) in which w appears as the last
        # token. Needed for orders < max_order when used as the recursive
        # backoff base. We precompute this once per order pair actually used.
        self._continuation_cache = {}

        # PERFORMANCE: precompute, for every order from 1..self.order, the
        # number of distinct words following each context -- i.e.
        # distinct_following_table[n][context] = |{w : (context,w) in counts[n]}|.
        # Built once here (O(number of n-grams), not per logprob() call) so
        # `_distinct_following` below is an O(1) dict lookup instead of a full
        # linear scan over the n-gram table on every single token scored.
        # This matters a great deal at real corpus scale (~1.2M tokens):
        # without this, KN perplexity would be impractically slow.
        self._distinct_following_table: Dict[int, Dict[Tuple[str, ...], int]] = {}
        for n in range(1, self.order + 1):
            table = defaultdict(set)
            for ngram in ngram_counts.counts[n]:
                prefix, last = ngram[:-1], ngram[-1]
                table[prefix].add(last)
            self._distinct_following_table[n] = {k: len(v) for k, v in table.items()}

    def _continuation_prob(self, word: str, order: int) -> float:
        """P_continuation(word) at the given order: fraction of distinct
        (order+1)-gram types that end in `word`, used as the lowest-order KN
        base distribution (standard interpolated KN definition)."""
        cache_key = order
        if cache_key not in self._continuation_cache:
            higher = self.nc.counts.get(order + 1, {})
            # distinct contexts (order-grams) that precede each distinct
            # continuation word
            continuation_counts = defaultdict(set)
            for ngram in higher:
                prefix, last = ngram[:-1], ngram[-1]
                continuation_counts[last].add(prefix)
            total_types = sum(len(v) for v in continuation_counts.values())
            self._continuation_cache[cache_key] = (
                {w: len(ctxs) for w, ctxs in continuation_counts.items()},
                total_types,
            )
        type_counts, total_types = self._continuation_cache[cache_key]
        if total_types == 0:
            return 1.0 / max(self.V, 1)
        return type_counts.get(word, 0) / total_types

    def _kn_recursive(self, context: Tuple[str, ...], word: str, order: int) -> float:
        """Standard recursive interpolated KN probability (returns a plain
        probability, not log), for a given order, recursing down to the
        continuation-based unigram base case."""
        if order == 1:
            return self._continuation_prob(word, 1)

        ctx = context[-(order - 1):] if order > 1 else ()
        ngram = ctx + (word,)
        c_ngram = self.nc.counts[order].get(ngram, 0)
        c_context = self.nc.context_counts[order].get(ctx, 0)

        if c_context == 0:
            # Never saw this context at all -- fall back entirely to lower order
            return self._kn_recursive(context, word, order - 1)

        # Number of distinct word types following this context (for the
        # normalizing backoff weight lambda(context))
        # We need: distinct continuations of `ctx` at this order.
        distinct_continuations = self._distinct_following(ctx, order)

        discounted = max(c_ngram - self.d, 0.0) / c_context
        backoff_weight = (self.d * distinct_continuations) / c_context
        lower = self._kn_recursive(context, word, order - 1)

        return discounted + backoff_weight * lower

    def _distinct_following(self, context: Tuple[str, ...], order: int) -> int:
        """Number of distinct words w such that (context + (w,)) was seen at
        this order -- used for the KN backoff normalizing weight. O(1)
        lookup into the table precomputed in __init__ (see PERFORMANCE note
        there) -- do not revert this to a linear scan."""
        return self._distinct_following_table.get(order, {}).get(context, 0)

    def logprob(self, context: Tuple[str, ...], word: str) -> float:
        p_kn = self._kn_recursive(context, word, self.order)
        p_uni = self._unigram_prob.get(word, self._unigram_floor)
        prob = self.lam * p_kn + (1 - self.lam) * p_uni
        return math.log(max(prob, 1e-12))


# ---------------------------------------------------------------------------
# Perplexity evaluation (shared by all 5 variants)
# ---------------------------------------------------------------------------

def perplexity(model, stream: List[str], order: int) -> float:
    """Compute perplexity of `model` over `stream`, using sliding windows of
    the given order. `stream` should already include <s>/</s> boundaries
    (i.e. it's the output of tokenizer.build_scoring_stream).

    Convention (PLAN.md 3.2): every token is scored INCLUDING </s>, EXCLUDING
    <s> as a predicted token (it only ever appears as context).
    """
    from tokenizer import BOS

    total_logprob = 0.0
    n_tokens = 0

    for i in range(len(stream)):
        word = stream[i]
        if word == BOS:
            continue  # never score <s> itself, it's context-only
        context_start = max(0, i - (order - 1))
        context = tuple(stream[context_start:i])
        total_logprob += model.logprob(context, word)
        n_tokens += 1

    avg_neg_logprob = -total_logprob / n_tokens
    return math.exp(avg_neg_logprob)


# ---------------------------------------------------------------------------
# KN lambda tuning (grid search on validation, per PLAN.md 3.4)
# ---------------------------------------------------------------------------

def tune_kn_lambda(ngram_counts: NgramCounts, order: int, val_stream: List[str],
                    d: float = 0.75, grid=None) -> Tuple[float, float]:
    """Grid search lam over `grid` (default {0.1,...,0.9}), minimizing
    validation perplexity. Returns (best_lam, best_val_ppl)."""
    if grid is None:
        grid = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

    best_lam, best_ppl = None, float("inf")
    for lam in grid:
        model = KneserNeyModel(ngram_counts, order, d=d, lam=lam)
        ppl = perplexity(model, val_stream, order)
        if ppl < best_ppl:
            best_lam, best_ppl = lam, ppl
    return best_lam, best_ppl


if __name__ == "__main__":
    # Smoke test on fabricated data -- NOT real DailyDialog, just verifies
    # the module runs end-to-end and produces sane (finite, >1) perplexities
    # before we touch Colab.
    import sys
    sys.path.insert(0, "/home/claude")
    from tokenizer import build_scoring_stream, Vocab, build_vocab, tokenize, join_dialogue

    fake_dialogues = [
        ["I like cats.", "Cats are nice.", "Do you like dogs too?"],
        ["I like dogs.", "Dogs are loyal.", "Cats are independent."],
        ["Do you like cats or dogs?", "I like both animals."],
        ["Cats are independent.", "Dogs are loyal and nice."],
    ] * 5  # repeat for slightly less sparse counts in this tiny smoke test

    token_lists = [tokenize(join_dialogue(d)) for d in fake_dialogues]
    vocab = build_vocab(token_lists, max_size=1000, min_count=1)
    stream = build_scoring_stream(fake_dialogues, vocab)

    print("Stream length:", len(stream), "Vocab size:", len(vocab))

    nc = NgramCounts(stream, max_order=4)

    print("\n--- Add-k (k=1), trigram ---")
    addk = AddKModel(nc, order=3, k=1.0)
    print("PPL:", perplexity(addk, stream, order=3))

    print("\n--- Good-Turing, trigram ---")
    gt = GoodTuringModel(nc, order=3, gt_max=5)
    print("PPL:", perplexity(gt, stream, order=3))

    print("\n--- Kneser-Ney, trigram (lam=1.0, i.e. pure KN, no unigram blend) ---")
    kn = KneserNeyModel(nc, order=3, d=0.75, lam=1.0)
    print("PPL:", perplexity(kn, stream, order=3))

    print("\n--- KN lambda tuning (using the same stream as 'validation' -- smoke test only) ---")
    best_lam, best_ppl = tune_kn_lambda(nc, order=3, val_stream=stream)
    print("Best lambda:", best_lam, "Best PPL:", best_ppl)

    print("\nAll smoke tests ran without error.")

Writing ngram_lm.py


In [8]:
import importlib
import tokenizer as tok
import ngram_lm as ngl
importlib.reload(tok)
importlib.reload(ngl)
import json, time, math, urllib.request
import pandas as pd
from datasets import Dataset, DatasetDict

print("Modules loaded, no Drive dependency.")

Modules loaded, no Drive dependency.


In [9]:
import urllib.request
import pandas as pd
from datasets import Dataset, DatasetDict

urls = {
    "train": "https://huggingface.co/datasets/daily_dialog/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet",
    "validation": "https://huggingface.co/datasets/daily_dialog/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet",
    "test": "https://huggingface.co/datasets/daily_dialog/resolve/refs%2Fconvert%2Fparquet/default/test/0000.parquet",
}

local_paths = {}
for split, url in urls.items():
    local_path = f"/content/daily_dialog_{split}.parquet"
    urllib.request.urlretrieve(url, local_path)
    local_paths[split] = local_path

dfs = {split: pd.read_parquet(path) for split, path in local_paths.items()}
for split, df in dfs.items():
    print(split, df.shape)

train (11118, 3)
validation (1000, 3)
test (1000, 3)


In [10]:
train_token_lists = [tok.tokenize(tok.join_dialogue(d)) for d in dfs['train']['dialog'].tolist()]
vocab = tok.build_vocab(train_token_lists, max_size=30000, min_count=2)
print("Vocab size:", len(vocab))

train_stream = tok.build_scoring_stream(dfs['train']['dialog'].tolist(), vocab)
val_stream = tok.build_scoring_stream(dfs['validation']['dialog'].tolist(), vocab)
test_stream = tok.build_scoring_stream(dfs['test']['dialog'].tolist(), vocab)
print("Stream lengths:", len(train_stream), len(val_stream), len(test_stream))

Vocab size: 12482
Stream lengths: 1201536 110872 108079


In [11]:
t0 = time.time()
nc = ngl.NgramCounts(train_stream, max_order=4)
print("NgramCounts built in", time.time() - t0, "s")

NgramCounts built in 5.155279159545898 s


In [12]:
orders = [3, 4]
results = []

for order in orders:
    for k in [1.0, 0.5, 0.1]:
        model = ngl.AddKModel(nc, order=order, k=k)
        test_ppl = ngl.perplexity(model, test_stream, order=order)
        results.append({"order": order, "variant": f"Add-k (k={k})", "test_ppl": test_ppl})
        print(f"[order={order}] Add-k (k={k}): test PPL = {test_ppl:.3f}")

    model = ngl.GoodTuringModel(nc, order=order, gt_max=5)
    test_ppl = ngl.perplexity(model, test_stream, order=order)
    results.append({"order": order, "variant": "Good-Turing", "test_ppl": test_ppl})
    print(f"[order={order}] Good-Turing: test PPL = {test_ppl:.3f}")

    best_lam, best_val_ppl = ngl.tune_kn_lambda(nc, order=order, val_stream=val_stream)
    kn_model = ngl.KneserNeyModel(nc, order=order, d=0.75, lam=best_lam)
    test_ppl = ngl.perplexity(kn_model, test_stream, order=order)
    results.append({"order": order, "variant": "Kneser-Ney", "test_ppl": test_ppl,
                     "val_ppl": best_val_ppl, "lambda": best_lam})
    print(f"[order={order}] Kneser-Ney: test PPL = {test_ppl:.3f} (val PPL={best_val_ppl:.3f}, lambda={best_lam})")

print("\n=== FULL RESULTS TABLE ===")
for r in results:
    print(r)

[order=3] Add-k (k=1.0): test PPL = 2038.165
[order=3] Add-k (k=0.5): test PPL = 1384.347
[order=3] Add-k (k=0.1): test PPL = 559.825
[order=3] Good-Turing: test PPL = 59.838
[order=3] Kneser-Ney: test PPL = 45.568 (val PPL=44.764, lambda=0.9)
[order=4] Add-k (k=1.0): test PPL = 5198.209
[order=4] Add-k (k=0.5): test PPL = 3962.366
[order=4] Add-k (k=0.1): test PPL = 1911.630
[order=4] Good-Turing: test PPL = 61.174
[order=4] Kneser-Ney: test PPL = 33.060 (val PPL=34.447, lambda=0.9)

=== FULL RESULTS TABLE ===
{'order': 3, 'variant': 'Add-k (k=1.0)', 'test_ppl': 2038.165370386408}
{'order': 3, 'variant': 'Add-k (k=0.5)', 'test_ppl': 1384.347126402998}
{'order': 3, 'variant': 'Add-k (k=0.1)', 'test_ppl': 559.8248788992261}
{'order': 3, 'variant': 'Good-Turing', 'test_ppl': 59.838287322390755}
{'order': 3, 'variant': 'Kneser-Ney', 'test_ppl': 45.56839098894333, 'val_ppl': 44.76430939881074, 'lambda': 0.9}
{'order': 4, 'variant': 'Add-k (k=1.0)', 'test_ppl': 5198.20866501254}
{'order': 4

In [13]:
best_kn_model = ngl.KneserNeyModel(nc, order=4, d=0.75, lam=0.9)
print("Best model: 4-gram Kneser-Ney, lambda=0.9")

Best model: 4-gram Kneser-Ney, lambda=0.9


In [14]:
BLIMP_PARADIGMS = [
    "anaphor_gender_agreement", "determiner_noun_agreement_1",
    "regular_plural_subject_verb_agreement_1", "wh_island", "npi_present_1",
    "left_branch_island_simple_question", "principle_A_domain_1",
    "drop_argument", "existential_there_quantifiers_1",
]

def load_blimp_paradigm(paradigm_name):
    url = f"https://huggingface.co/datasets/nyu-mll/blimp/resolve/main/{paradigm_name}/train-00000-of-00001.parquet"
    local_path = f"/content/blimp_{paradigm_name}.parquet"
    urllib.request.urlretrieve(url, local_path)
    return pd.read_parquet(local_path)

blimp_dfs = {p: load_blimp_paradigm(p) for p in BLIMP_PARADIGMS}
print("Loaded", len(blimp_dfs), "paradigms")

Loaded 9 paradigms


In [15]:
def score_sentence_ngram(model, sentence, order):
    toks = tok.tokenize(sentence)
    toks = vocab.encode_str(toks)
    seq = [tok.BOS] + toks + [tok.EOS]
    total_logprob, n_scored = 0.0, 0
    for i in range(len(seq)):
        if seq[i] == tok.BOS:
            continue
        context_start = max(0, i - (order - 1))
        context = tuple(seq[context_start:i])
        total_logprob += model.logprob(context, seq[i])
        n_scored += 1
    return total_logprob / n_scored

def evaluate_blimp(model, order, paradigms_dict):
    per_paradigm = {}
    total_correct, total_pairs = 0, 0
    for paradigm_name, df in paradigms_dict.items():
        correct = 0
        for _, row in df.iterrows():
            lp_good = score_sentence_ngram(model, row['sentence_good'], order)
            lp_bad = score_sentence_ngram(model, row['sentence_bad'], order)
            if lp_good > lp_bad:
                correct += 1
        acc = correct / len(df)
        per_paradigm[paradigm_name] = acc
        total_correct += correct
        total_pairs += len(df)
        print(f"{paradigm_name}: {acc:.3f} ({correct}/{len(df)})")
    overall_acc = total_correct / total_pairs
    print(f"\nOverall BLiMP accuracy: {overall_acc:.3f} ({total_correct}/{total_pairs})")
    return per_paradigm, overall_acc

per_paradigm_acc, overall_acc = evaluate_blimp(best_kn_model, order=4, paradigms_dict=blimp_dfs)

anaphor_gender_agreement: 0.310 (310/1000)
determiner_noun_agreement_1: 0.533 (533/1000)
regular_plural_subject_verb_agreement_1: 0.462 (462/1000)
wh_island: 0.618 (618/1000)
npi_present_1: 0.464 (464/1000)
left_branch_island_simple_question: 0.286 (286/1000)
principle_A_domain_1: 1.000 (1000/1000)
drop_argument: 0.465 (465/1000)
existential_there_quantifiers_1: 0.910 (910/1000)

Overall BLiMP accuracy: 0.561 (5048/9000)


In [16]:
print("=== TASK 1 FINAL SUMMARY ===")
print("Best model: 4-gram Kneser-Ney (lambda=0.9)")
print("Test perplexity: 33.060")
print("Validation perplexity: 34.447")
print(f"BLiMP overall accuracy: {overall_acc:.3f}")
print("\nFull per-variant perplexity table and per-paradigm BLiMP results are printed above.")
print("This notebook is self-contained and requires no Google Drive access to run.")

=== TASK 1 FINAL SUMMARY ===
Best model: 4-gram Kneser-Ney (lambda=0.9)
Test perplexity: 33.060
Validation perplexity: 34.447
BLiMP overall accuracy: 0.561

Full per-variant perplexity table and per-paradigm BLiMP results are printed above.
This notebook is self-contained and requires no Google Drive access to run.
